# Day 13: NGBoost Probability-Quality Evaluation

Goal: determine whether NGBoost is calibrated well enough to be the primary probability signal. This report evaluates probability quality, not point-prediction accuracy.

The target remains `forecast_error = actual official high - NWS forecast high`. All final metrics use chronological out-of-sample rows only.

## Input Mapping

- NGBoost distribution parameters: `outputs/ngboost_distribution_params_v0.csv` with `mu`, `sigma`, `forecast_error`, timestamps, and `split`.
- NGBoost bucket probabilities: `outputs/ngboost_bucket_probs_v0.csv`, long format with one row per `row_id` and market bucket. Bucket names are final-temperature labels that move with `forecast_high`, so the Day 13 market-bucket diagnostic evaluates fixed `bucket_index` positions as `market_bucket_0` through `market_bucket_5`.
- Empirical Day 9 baseline: `outputs/day9_empirical_baseline/empirical_baseline_predictions.csv`, fixed forecast-error intervals. The baseline comparison therefore evaluates NGBoost on those same forecast-error intervals for a fair same-row comparison.
- Modeling rows: `data/processed/modeling_rows_v1.csv`, used only to recover diagnostics such as `season` when not present in the prediction artifact.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

paths = {
    "ngboost_params": ROOT / "outputs" / "ngboost_distribution_params_v0.csv",
    "ngboost_bucket_probs": ROOT / "outputs" / "ngboost_bucket_probs_v0.csv",
    "empirical_baseline": ROOT / "outputs" / "day9_empirical_baseline" / "empirical_baseline_predictions.csv",
    "coverage_report": ROOT / "outputs" / "coverage_report.csv",
    "residual_summary": ROOT / "outputs" / "standardized_residual_summary.csv",
    "bucket_brier": ROOT / "outputs" / "bucket_brier_scores.csv",
    "calibration_tables": ROOT / "outputs" / "calibration_tables.csv",
    "coverage_by_group": ROOT / "outputs" / "coverage_by_group.csv",
    "evaluation_report": ROOT / "outputs" / "ngboost_evaluation_report.csv",
    "pit_histogram": ROOT / "outputs" / "figures" / "pit_histogram.png",
    "coverage_by_hour": ROOT / "outputs" / "figures" / "coverage_by_hour.png",
    "coverage_by_season": ROOT / "outputs" / "figures" / "coverage_by_season.png",
}

row_counts = []
for name, path in paths.items():
    if path.suffix == ".csv":
        row_counts.append({"artifact": name, "path": str(path.relative_to(ROOT)), "rows": len(pd.read_csv(path))})
    else:
        row_counts.append({"artifact": name, "path": str(path.relative_to(ROOT)), "rows": None})
pd.DataFrame(row_counts)

## Validation Checks

The runner validates sigma positivity, finite distribution parameters, bucket probability sums, realized bucket labels, stable key alignment, and chronological validation/test ordering before metrics are written. It does not retrain or add features; `modeling_rows_v1.csv` is used only for diagnostics such as `season`.

In [ ]:
params = pd.read_csv(paths["ngboost_params"])
bucket_long = pd.read_csv(paths["ngboost_bucket_probs"])
baseline = pd.read_csv(paths["empirical_baseline"])

if "row_id" not in params.columns:
    params.insert(0, "row_id", range(len(params)))

bucket_sums = bucket_long.groupby("row_id")["probability"].sum()
dates = pd.to_datetime(params["date"])
validation_dates = dates[params["split"] == "validation"]
test_dates = dates[params["split"] == "test"]

key_columns = ["date", "prediction_time", "location", "forecast_horizon_hours"]
checks = pd.DataFrame([
    {"check": "required NGBoost columns present", "passed": {"forecast_error", "mu", "sigma", "split"}.issubset(params.columns)},
    {"check": "no missing stable key columns", "passed": params[key_columns].notna().all().all()},
    {"check": "mu finite", "passed": pd.to_numeric(params["mu"], errors="coerce").notna().all()},
    {"check": "sigma positive", "passed": (pd.to_numeric(params["sigma"], errors="coerce") > 0).all()},
    {"check": "bucket probabilities sum to 1", "passed": ((bucket_sums - 1).abs().max() <= 1e-6)},
    {"check": "validation before test", "passed": validation_dates.max() < test_dates.min()},
    {"check": "baseline rows match NGBoost test rows", "passed": len(baseline) == (params["split"] == "test").sum()},
])
checks

## NGBoost Distribution Scoring

Continuous distribution metrics use the Normal forecast-error distribution emitted by NGBoost. NLL is a density log score, so it is only reported for NGBoost, not for the empirical baseline unless that baseline emits a comparable density.

In [ ]:
evaluation_report = pd.read_csv(paths["evaluation_report"])
residual_summary = pd.read_csv(paths["residual_summary"])

display(evaluation_report)
display(residual_summary)
display(Image(filename=str(paths["pit_histogram"])))

## Prediction Interval Coverage

If actual coverage is below expected coverage, the predictive distribution is overconfident. If actual coverage is above expected coverage, it is underconfident.

In [ ]:
coverage = pd.read_csv(paths["coverage_report"])
coverage

## Bucket Probability Scoring

The market-bucket table below evaluates the saved NGBoost bucket probabilities by fixed market bucket position. This avoids mixing moving final-temperature labels across different forecast highs.

In [ ]:
bucket_brier = pd.read_csv(paths["bucket_brier"])
display(bucket_brier)

worst_buckets = bucket_brier.sort_values("brier_score", ascending=False).head(8)
worst_buckets

## Calibration Curves

Selected curves include the most common market bucket, the worst Brier bucket, and the lower/upper market tails when identifiable.

In [ ]:
calibration_tables = pd.read_csv(paths["calibration_tables"])
display(calibration_tables.head(20))

for image_path in sorted((ROOT / "outputs" / "figures").glob("calibration_bucket_*.png")):
    display(Image(filename=str(image_path)))

## Diagnostics By Hour And Season

Small groups are retained and flagged with `enough_sample` instead of being silently dropped.

In [ ]:
coverage_by_group = pd.read_csv(paths["coverage_by_group"])
display(coverage_by_group.head(40))
display(Image(filename=str(paths["coverage_by_hour"])))
if paths["coverage_by_season"].exists():
    display(Image(filename=str(paths["coverage_by_season"])))

## Empirical Baseline Comparison

The Day 9 empirical baseline only emits forecast-error interval probabilities, not a continuous density. The fair comparison is therefore bucket Brier and interval log loss on the exact same test rows and the exact same Day 9 forecast-error intervals.

In [ ]:
evaluation_report

## Interpretation Rules

- If actual coverage < expected coverage, NGBoost is overconfident.
- If actual coverage > expected coverage, NGBoost is underconfident.
- If standardized residual std > 1, sigma is likely too small.
- If standardized residual std < 1, sigma is likely too large.
- If the PIT histogram is U-shaped, the distribution is too narrow.
- If the PIT histogram is hump-shaped, the distribution is too wide.
- If the PIT histogram is skewed, the model has directional bias.
- If Brier is okay but interval log loss is bad, the model sometimes assigns dangerously low probability to realized events.
- If empirical baseline beats NGBoost, document it honestly; it may mean sample size is small or simple historical forecast-error structure is strong.

## Conclusion

NGBoost is not calibrated enough to be the sole primary probability signal yet. It is useful and materially beats the Day 9 empirical baseline on the same test rows and same forecast-error interval schema: test interval log loss is 0.955 for NGBoost versus 1.489 for the empirical baseline, and mean bucket Brier is 0.102 versus 0.154. That means NGBoost has a real probability signal.

The calibration problem is still too large to ignore. On the test split, 80% interval coverage is 72.65%, so the model is overconfident out of sample. Test standardized residual std is 1.30, which also says sigma is likely too small. Residual mean is -0.26, indicating directional bias: realized forecast errors are more negative than the model distribution expects. Market bucket diagnostics show tail weakness too, especially lower-tail underprediction in `market_bucket_0` on test rows.

Next improvement should be calibration and tail diagnostics, not feature engineering or model optimization today: recalibrate sigma, inspect PIT skew by hour/season/horizon, and consider split-aware post-hoc calibration before using NGBoost as the primary tradable probability signal.